In [1]:
import pandas as pd 
import os
import numpy as np 
import openpyxl

In [2]:
print(os.getcwd())

c:\Users\User\.vscode\analysis-practice\KAIST_25fall\assetpricing\Assignment 2


In [3]:
monthly_beta_data = pd.read_csv("monthly_beta_data.csv")
assignment1_data = pd.read_csv("assignment1_data.csv")

In [4]:
monthly_beta_data['t'] = pd.to_datetime(monthly_beta_data['t'], format='%Y%m%d')
monthly_beta_data.head()
monthly_beta_data.columns = ['permno', 'b_mkt', 't']

In [5]:
print(assignment1_data.dtypes)
assignment1_data.head()

permno           int64
date             int64
year             int64
exchcd           int64
siccd          float64
retadj         float64
eretadj        float64
altprc_lag1    float64
ME_lag1        float64
ME_Jun         float64
BM             float64
dtype: object


,permno,date,year,exchcd,siccd,retadj,eretadj,altprc_lag1,ME_lag1,ME_Jun,BM
0,10001,19870731,1987,3,4920.0,0.021277,0.016677,5.875,5.822125,5.822125,1.014415
1,10001,19870831,1987,3,4920.0,0.083333,0.078633,6.000,5.946000,5.822125,1.014415
2,10001,19870930,1987,3,4920.0,-0.022308,-0.026808,6.500,6.441500,5.822125,1.014415
3,10001,19871030,1987,3,4920.0,0.020000,0.014000,6.250,6.200000,5.822125,1.014415
4,10001,19871130,1987,3,4920.0,-0.029412,-0.032912,6.375,6.324000,5.822125,1.014415


In [6]:
#generate the month-end date of each monthly stock observation 
assignment1_data['date'] = pd.to_datetime(assignment1_data['date'], format='%Y%m%d')
assignment1_data['t'] = assignment1_data['date'] + pd.offsets.MonthEnd(0)
assignment1_data.head()
monthly_stock_data = assignment1_data.copy()

In [7]:
# add beta info to the monthly stock data set 
monthly_stock_data2 = monthly_stock_data.merge(monthly_beta_data , on=['permno', 't'], how = 'inner')
monthly_stock_data2 = monthly_stock_data2.dropna(subset='b_mkt')

In [8]:
# generate mktcap_CPI and size variables 
cpi = pd.read_excel('CPIAUCSL.xls',
                    sheet_name="Sheet1",
                    header=10,
)
cpi.columns = ['cpi_date', 'cpi']
# load cpi data 
cpi.head()
print(cpi.dtypes)

cpi_date    datetime64[ns]
cpi                float64
dtype: object


In [9]:
#cpi in june of each year 
cpi['t'] = cpi['cpi_date'].dt.year
query = cpi['cpi_date'].dt.month == 6
cpi_jun = cpi.loc[query,]
cpi_jun.head()

# cpi in dec, 2012
cpi_2012 = 231.221 

In [10]:
# calculate mktcap_cpi , size , and log_bm variable 
monthly_stock_data2.head()

query = monthly_stock_data2['date'].dt.month > 6 
monthly_stock_data2['t'] = np.where(monthly_stock_data2['date'].dt.month > 6, 
                                    monthly_stock_data2['date'].dt.year,
                                    monthly_stock_data2['date'].dt.year - 1)

monthly_stock_data3 = monthly_stock_data2.merge(
    cpi_jun,
    on = ['t'],
    how='left'
)

monthly_stock_data3['ME_Jun_CPI'] = monthly_stock_data3['ME_Jun']/monthly_stock_data3['cpi'] * cpi_2012
monthly_stock_data3['size'] = np.log(monthly_stock_data3['ME_Jun'])
monthly_stock_data3['size_CPI'] = np.log(monthly_stock_data3['ME_Jun_CPI'])
monthly_stock_data3['log_BM'] = np.log(monthly_stock_data3['BM'])

monthly_stock_data3.head()

,permno,date,year,exchcd,siccd,retadj,eretadj,altprc_lag1,ME_lag1,ME_Jun,BM,t,b_mkt,cpi_date,cpi,ME_Jun_CPI,size,size_CPI,log_BM
0,10001,1987-07-31,1987,3,4920.0,0.021277,0.016677,5.875,5.822125,5.822125,1.014415,1987,0.3545,1987-06-01,113.5,11.860771,1.761665,2.473236,0.014313
1,10001,1987-08-31,1987,3,4920.0,0.083333,0.078633,6.000,5.946000,5.822125,1.014415,1987,0.3862,1987-06-01,113.5,11.860771,1.761665,2.473236,0.014313
2,10001,1987-09-30,1987,3,4920.0,-0.022308,-0.026808,6.500,6.441500,5.822125,1.014415,1987,0.5362,1987-06-01,113.5,11.860771,1.761665,2.473236,0.014313
3,10001,1987-10-30,1987,3,4920.0,0.020000,0.014000,6.250,6.200000,5.822125,1.014415,1987,0.6279,1987-06-01,113.5,11.860771,1.761665,2.473236,0.014313
4,10001,1987-11-30,1987,3,4920.0,-0.029412,-0.032912,6.375,6.324000,5.822125,1.014415,1987,0.1559,1987-06-01,113.5,11.860771,1.761665,2.473236,0.014313


In [11]:
# winsorize stock characteristic variables 
monthly_stock_data3=monthly_stock_data3.rename(columns={
    'b_mkt': 'b_mkt_o',
    'size': 'size_o',
    'size_CPI': 'size_CPI_o',
    'log_BM': 'log_BM_o',
    'BM': 'BM_o'
})

monthly_stock_data3.head()

,permno,date,year,exchcd,siccd,retadj,eretadj,altprc_lag1,ME_lag1,ME_Jun,BM_o,t,b_mkt_o,cpi_date,cpi,ME_Jun_CPI,size_o,size_CPI_o,log_BM_o
0,10001,1987-07-31,1987,3,4920.0,0.021277,0.016677,5.875,5.822125,5.822125,1.014415,1987,0.3545,1987-06-01,113.5,11.860771,1.761665,2.473236,0.014313
1,10001,1987-08-31,1987,3,4920.0,0.083333,0.078633,6.000,5.946000,5.822125,1.014415,1987,0.3862,1987-06-01,113.5,11.860771,1.761665,2.473236,0.014313
2,10001,1987-09-30,1987,3,4920.0,-0.022308,-0.026808,6.500,6.441500,5.822125,1.014415,1987,0.5362,1987-06-01,113.5,11.860771,1.761665,2.473236,0.014313
3,10001,1987-10-30,1987,3,4920.0,0.020000,0.014000,6.250,6.200000,5.822125,1.014415,1987,0.6279,1987-06-01,113.5,11.860771,1.761665,2.473236,0.014313
4,10001,1987-11-30,1987,3,4920.0,-0.029412,-0.032912,6.375,6.324000,5.822125,1.014415,1987,0.1559,1987-06-01,113.5,11.860771,1.761665,2.473236,0.014313


In [12]:
#calculate 0.5% and 99.5% level of each cahracteristic variable on a monthly basis 
bounds = monthly_stock_data3[[ 'date', 'b_mkt_o', 'size_o', 'size_CPI_o', 'log_BM_o', 'BM_o']]
bounds = bounds.groupby('date').quantile([0.005,0.995]).unstack().rename(columns={
    0.005 : '0_5',
    0.995 : '99_5'
}).reset_index()

bounds.columns = [f"{col[0].replace('_o', '')}_{col[1]}" for col in bounds.columns]
bounds=bounds.rename(
    columns={'date_' : 'date'}
)
bounds.head()

,date,b_mkt_0_5,b_mkt_99_5,size_0_5,size_99_5,size_CPI_0_5,size_CPI_99_5,log_BM_0_5,log_BM_99_5,BM_0_5,BM_99_5
0,1963-02-28,0.236940,1.467132,2.614917,7.279823,4.650118,9.315024,-1.425338,0.293109,0.240473,1.340889
1,1963-03-29,0.233056,1.463296,2.614917,7.279823,4.650118,9.315024,-1.425338,0.293109,0.240473,1.340889
2,1963-04-30,0.238704,1.475008,2.614917,7.279823,4.650118,9.315024,-1.425338,0.293109,0.240473,1.340889
3,1963-05-31,0.237316,1.491400,2.614917,7.279823,4.650118,9.315024,-1.425338,0.293109,0.240473,1.340889
4,1963-06-28,0.162260,1.219948,2.614917,7.279823,4.650118,9.315024,-1.425338,0.293109,0.240473,1.340889


In [13]:
# merge the bounds with the monthly stock data and winsorize characteristic variables 

monthly_stock_data4 = monthly_stock_data3.merge(bounds , on = 'date', how='left') #full join 
monthly_stock_data4.head()

,permno,date,year,exchcd,siccd,retadj,eretadj,altprc_lag1,ME_lag1,ME_Jun,...,b_mkt_0_5,b_mkt_99_5,size_0_5,size_99_5,size_CPI_0_5,size_CPI_99_5,log_BM_0_5,log_BM_99_5,BM_0_5,BM_99_5
0,10001,1987-07-31,1987,3,4920.0,0.021277,0.016677,5.875,5.822125,5.822125,...,-0.621340,1.839786,-0.306598,9.694866,0.404973,10.406437,-3.929214,1.518371,0.019659,4.564804
1,10001,1987-08-31,1987,3,4920.0,0.083333,0.078633,6.000,5.946000,5.822125,...,-0.707710,1.806320,-0.294948,9.694277,0.416623,10.405848,-3.928624,1.511336,0.019671,4.532790
2,10001,1987-09-30,1987,3,4920.0,-0.022308,-0.026808,6.500,6.441500,5.822125,...,-0.812136,1.765856,-0.269555,9.695287,0.442016,10.406858,-3.929635,1.511714,0.019651,4.534509
3,10001,1987-10-30,1987,3,4920.0,0.020000,0.014000,6.250,6.200000,5.822125,...,-0.820385,1.793791,-0.233461,9.695792,0.478110,10.407363,-3.953642,1.494390,0.019189,4.456757
4,10001,1987-11-30,1987,3,4920.0,-0.029412,-0.032912,6.375,6.324000,5.822125,...,-0.330410,1.955328,-0.192142,9.697811,0.519429,10.409382,-3.959860,1.496807,0.019071,4.467597


In [14]:
columns = ['b_mkt', 'size', 'size_CPI', 'log_BM', 'BM']

for c in columns:
    monthly_stock_data4[c] = np.where(
        monthly_stock_data4[f"{c}_o"] <  monthly_stock_data4[f"{c}_0_5"] ,
        monthly_stock_data4[f"{c}_0_5"],
        np.where(
            monthly_stock_data4[f"{c}_o"] >  monthly_stock_data4[f"{c}_99_5"],
            monthly_stock_data4[f"{c}_99_5"],
            monthly_stock_data4[f"{c}_o"]
        ) 
    )
    monthly_stock_data4 = monthly_stock_data4.drop(columns=[f"{c}_o", f"{c}_0_5",f"{c}_99_5"])

monthly_stock_data4.head()

,permno,date,year,exchcd,siccd,retadj,eretadj,altprc_lag1,ME_lag1,ME_Jun,t,cpi_date,cpi,ME_Jun_CPI,b_mkt,size,size_CPI,log_BM,BM
0,10001,1987-07-31,1987,3,4920.0,0.021277,0.016677,5.875,5.822125,5.822125,1987,1987-06-01,113.5,11.860771,0.3545,1.761665,2.473236,0.014313,1.014415
1,10001,1987-08-31,1987,3,4920.0,0.083333,0.078633,6.000,5.946000,5.822125,1987,1987-06-01,113.5,11.860771,0.3862,1.761665,2.473236,0.014313,1.014415
2,10001,1987-09-30,1987,3,4920.0,-0.022308,-0.026808,6.500,6.441500,5.822125,1987,1987-06-01,113.5,11.860771,0.5362,1.761665,2.473236,0.014313,1.014415
3,10001,1987-10-30,1987,3,4920.0,0.020000,0.014000,6.250,6.200000,5.822125,1987,1987-06-01,113.5,11.860771,0.6279,1.761665,2.473236,0.014313,1.014415
4,10001,1987-11-30,1987,3,4920.0,-0.029412,-0.032912,6.375,6.324000,5.822125,1987,1987-06-01,113.5,11.860771,0.1559,1.761665,2.473236,0.014313,1.014415


In [15]:
# calculate summary statistics 
stats_by_col = ['b_mkt', 'size', 'size_CPI', 'BM', 'log_BM']
def p25(x):
    return x.quantile(0.25)

def p75(x):
    return x.quantile(0.75)

results = []
for col in stats_by_col:
    test = monthly_stock_data4.groupby('date')[col].agg([
        'mean',
        'std',
        pd.Series.skew,
        pd.Series.kurt,
        'min',
        p25,
        'median',
        p75,
        'max',
        'count'
    ])
    test['variable'] = col
    results.append(test)

stats = pd.concat(results)
stats.head()

,mean,std,skew,kurt,min,p25,median,p75,max,count,variable
date,,,,,,,,,,,
1963-02-28,0.748108,0.312256,0.204130,-0.894235,0.236940,0.4890,0.7534,1.0367,1.467132,73,b_mkt
1963-03-29,0.750501,0.309996,0.186560,-0.901166,0.233056,0.4977,0.7647,1.0252,1.463296,73,b_mkt
1963-04-30,0.750854,0.308871,0.201838,-0.869832,0.238704,0.4989,0.7652,1.0124,1.475008,73,b_mkt
1963-05-31,0.754791,0.311769,0.226500,-0.825335,0.237316,0.5091,0.7646,1.0303,1.491400,73,b_mkt
1963-06-28,0.590623,0.233035,0.417155,-0.062010,0.162260,0.4227,0.5749,0.7418,1.219948,73,b_mkt


In [16]:
# reorder the variables 

var_orders = ['b_mkt', 'size', 'size_CPI', 'BM', 'log_BM']

stats = stats.groupby('variable').mean()
# stats.index = var_orders
stats = stats.loc[var_orders]
stats = stats.round(3)

In [17]:
#calculate size breakpoints as 20th, 40th, 60th, and 80th size percentiles among NYSE stocks in each month

query = monthly_stock_data4['exchcd'].isin([1,31])
size_breakpoints = monthly_stock_data4.loc[query,].groupby('date')['size'].quantile([0.2,0.4,0.6,0.8]).unstack()
size_breakpoints.columns = [f"size_{col}" for col in size_breakpoints.columns]
size_breakpoints = size_breakpoints.reset_index()

In [18]:
# merge the size breakpoints with the monthly stock data and define size sorted portfolios

monthly_stock_data5 = monthly_stock_data4.merge(size_breakpoints , on = 'date', how = 'left')
monthly_stock_data5['p1'] = np.where(monthly_stock_data5['size'] < monthly_stock_data5['size_0.2'], 1, 
                                     np.where(monthly_stock_data5['size'] < monthly_stock_data5['size_0.4'], 2 , np.where(
                                         monthly_stock_data5['size'] < monthly_stock_data5['size_0.6'] , 3 , np.where(
                                             monthly_stock_data5['size'] < monthly_stock_data5['size_0.8'] ,4 ,5
                                         )
                                     )))
monthly_stock_data5.head()

,permno,date,year,exchcd,siccd,retadj,eretadj,altprc_lag1,ME_lag1,ME_Jun,...,b_mkt,size,size_CPI,log_BM,BM,size_0.2,size_0.4,size_0.6,size_0.8,p1
0,10001,1987-07-31,1987,3,4920.0,0.021277,0.016677,5.875,5.822125,5.822125,...,0.3545,1.761665,2.473236,0.014313,1.014415,4.867511,5.760800,6.733022,7.695534,1
1,10001,1987-08-31,1987,3,4920.0,0.083333,0.078633,6.000,5.946000,5.822125,...,0.3862,1.761665,2.473236,0.014313,1.014415,4.872645,5.763480,6.734313,7.695303,1
2,10001,1987-09-30,1987,3,4920.0,-0.022308,-0.026808,6.500,6.441500,5.822125,...,0.5362,1.761665,2.473236,0.014313,1.014415,4.857987,5.760245,6.737538,7.695649,1
3,10001,1987-10-30,1987,3,4920.0,0.020000,0.014000,6.250,6.200000,5.822125,...,0.6279,1.761665,2.473236,0.014313,1.014415,4.854523,5.751724,6.734313,7.695807,1
4,10001,1987-11-30,1987,3,4920.0,-0.029412,-0.032912,6.375,6.324000,5.822125,...,0.1559,1.761665,2.473236,0.014313,1.014415,4.854982,5.751167,6.733269,7.700662,1


In [19]:
#calculate BM breakpoints as 20th, 40th, 60th and 80th BM percentiles among all stocks in each size sorted portfolio in each month 
bm_breakpoints = monthly_stock_data5.groupby(['date', 'p1'])['BM'].quantile([0.2,0.4,0.6,0.8]).unstack()
bm_breakpoints.columns = [f"bm_{col}" for col in bm_breakpoints.columns]
bm_breakpoints = bm_breakpoints.reset_index()
bm_breakpoints.head()

,date,p1,bm_0.2,bm_0.4,bm_0.6,bm_0.8
0,1963-02-28,1,0.438841,0.476999,0.506397,0.574200
1,1963-02-28,2,0.358932,0.408520,0.471159,0.563319
2,1963-02-28,3,0.309574,0.348930,0.447809,0.541939
3,1963-02-28,4,0.362776,0.401136,0.435341,0.481914
4,1963-02-28,5,0.373084,0.456510,0.487770,0.526183


In [20]:
#merge the BM breakpoints with the monthly stock data and define BM sorted portfolios in each size sorted portfolio 

monthly_stock_data6 = monthly_stock_data5.merge(bm_breakpoints, on= ['date', 'p1'], how='left')

monthly_stock_data6['p2'] = np.where(monthly_stock_data6['BM'] < monthly_stock_data6['bm_0.2'], 1, 
                                     np.where(monthly_stock_data6['BM'] < monthly_stock_data6['bm_0.4'], 2 , np.where(
                                         monthly_stock_data6['BM'] < monthly_stock_data6['bm_0.6'] , 3 , np.where(
                                             monthly_stock_data6['BM'] < monthly_stock_data6['bm_0.8'] ,4 ,5
                                         )
                                     )))
monthly_stock_data6.head()

,permno,date,year,exchcd,siccd,retadj,eretadj,altprc_lag1,ME_lag1,ME_Jun,...,size_0.2,size_0.4,size_0.6,size_0.8,p1,bm_0.2,bm_0.4,bm_0.6,bm_0.8,p2
0,10001,1987-07-31,1987,3,4920.0,0.021277,0.016677,5.875,5.822125,5.822125,...,4.867511,5.760800,6.733022,7.695534,1,0.299361,0.550817,0.795356,1.166487,4
1,10001,1987-08-31,1987,3,4920.0,0.083333,0.078633,6.000,5.946000,5.822125,...,4.872645,5.763480,6.734313,7.695303,1,0.297190,0.545433,0.791066,1.162875,4
2,10001,1987-09-30,1987,3,4920.0,-0.022308,-0.026808,6.500,6.441500,5.822125,...,4.857987,5.760245,6.737538,7.695649,1,0.297447,0.542103,0.788343,1.158233,4
3,10001,1987-10-30,1987,3,4920.0,0.020000,0.014000,6.250,6.200000,5.822125,...,4.854523,5.751724,6.734313,7.695807,1,0.296330,0.536852,0.783080,1.156812,4
4,10001,1987-11-30,1987,3,4920.0,-0.029412,-0.032912,6.375,6.324000,5.822125,...,4.854982,5.751167,6.733269,7.700662,1,0.295347,0.534196,0.779948,1.155792,4


In [21]:
# save the final data set in a local folder 
assignment2_data = monthly_stock_data6
assignment2_data.head(25)

,permno,date,year,exchcd,siccd,retadj,eretadj,altprc_lag1,ME_lag1,ME_Jun,...,size_0.2,size_0.4,size_0.6,size_0.8,p1,bm_0.2,bm_0.4,bm_0.6,bm_0.8,p2
0,10001,1987-07-31,1987,3,4920.0,0.021277,0.016677,5.8750,5.822125,5.822125,...,4.867511,5.760800,6.733022,7.695534,1,0.299361,0.550817,0.795356,1.166487,4
1,10001,1987-08-31,1987,3,4920.0,0.083333,0.078633,6.0000,5.946000,5.822125,...,4.872645,5.763480,6.734313,7.695303,1,0.297190,0.545433,0.791066,1.162875,4
2,10001,1987-09-30,1987,3,4920.0,-0.022308,-0.026808,6.5000,6.441500,5.822125,...,4.857987,5.760245,6.737538,7.695649,1,0.297447,0.542103,0.788343,1.158233,4
3,10001,1987-10-30,1987,3,4920.0,0.020000,0.014000,6.2500,6.200000,5.822125,...,4.854523,5.751724,6.734313,7.695807,1,0.296330,0.536852,0.783080,1.156812,4
4,10001,1987-11-30,1987,3,4920.0,-0.029412,-0.032912,6.3750,6.324000,5.822125,...,4.854982,5.751167,6.733269,7.700662,1,0.295347,0.534196,0.779948,1.155792,4
5,10001,1987-12-31,1987,3,4920.0,-0.033535,-0.037435,6.1875,6.138000,5.822125,...,4.854887,5.742646,6.730965,7.695933,1,0.295783,0.535073,0.780860,1.152505,4
6,10001,1988-01-29,1988,3,4920.0,0.063830,0.060930,5.8750,5.828000,5.822125,...,4.850824,5.724956,6.726545,7.695649,1,0.296034,0.536690,0.781823,1.152908,4
7,10001,1988-02-29,1988,3,4920.0,0.080000,0.075400,6.2500,6.200000,5.822125,...,4.848167,5.717277,6.718700,7.695534,1,0.296250,0.536794,0.782696,1.152483,4
8,10001,1988-03-31,1988,3,4920.0,-0.076296,-0.080696,6.7500,6.696000,5.822125,...,4.846802,5.712686,6.724357,7.695764,1,0.297632,0.538786,0.782405,1.150653,4
9,10001,1988-04-29,1988,3,4920.0,0.030612,0.026012,6.1250,6.076000,5.822125,...,4.846249,5.716129,6.729978,7.695807,1,0.299396,0.540160,0.786375,1.154910,4


In [22]:
assignment2_data.columns

Index(['permno', 'date', 'year', 'exchcd', 'siccd', 'retadj', 'eretadj',
       'altprc_lag1', 'ME_lag1', 'ME_Jun', 't', 'cpi_date', 'cpi',
       'ME_Jun_CPI', 'b_mkt', 'size', 'size_CPI', 'log_BM', 'BM', 'size_0.2',
       'size_0.4', 'size_0.6', 'size_0.8', 'p1', 'bm_0.2', 'bm_0.4', 'bm_0.6',
       'bm_0.8', 'p2'],
      dtype='object')

In [23]:
assignment2_data = assignment2_data[['permno', 'date', 'b_mkt', 'size', 'size_CPI', 'log_BM', 'BM', 'size_0.2',
       'size_0.4', 'size_0.6', 'size_0.8',  'bm_0.2', 'bm_0.4', 'bm_0.6',
       'bm_0.8', 'p1', 'p2']]
assignment2_data = assignment2_data.sort_values(by=['date', 'permno'])

In [24]:
assignment2_data = assignment2_data.round(3)
assignment2_data['date'] = pd.to_datetime(assignment2_data['date']).dt.strftime("%Y-%m-%d")

In [25]:
#calculate the time-series average number of stocks in each portfolio 
nstocks_per_p = monthly_stock_data6.groupby(['date', 'p1', 'p2'])['permno'].nunique().reset_index().rename(columns={'permno':'nstocks'})
nstocks_per_p = nstocks_per_p.groupby(['p1', 'p2'])['nstocks'].mean().reset_index(name='ave_nstocks')


In [26]:
nstocks_per_p = nstocks_per_p.pivot(index='p1', columns='p2', values='ave_nstocks')
nstocks_per_p.columns = [f"p2_{col}" for col in nstocks_per_p.columns]
nstocks_per_p = nstocks_per_p.round(3)
nstocks_per_p.head()

,p2_1,p2_2,p2_3,p2_4,p2_5
p1,,,,,
1,391.222,390.810,390.816,390.810,391.439
2,94.871,94.472,94.444,94.472,95.098
3,67.142,66.705,66.735,66.705,67.337
4,56.035,55.684,55.679,55.684,56.259
5,50.339,49.920,49.917,49.920,50.528
